In [1]:
"""
STEP 1: Load All OpenFOAM Cases
==================================
Extracts velocity profiles from all configurations and saves to pickle file.
"""

from setup_paths import SRC_DIR, DATA_DIR, RAW_DATA_DIR, PROCESSED_DATA_DIR 
import sys
import numpy as np
import pickle
import os

from OpenFOAM_reader import extract_profile_from_openfoam




In [2]:
# ============================================
# CONFIGURATION
# ============================================

# Definition of the cases (8 for now)
CASES = {
    'U1_uniform': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1',  
        'U_ref': 1.0,
        'profile_type': 0,                  # 0=uniform, 1=ABL+const, 2=ABL+free, 3=ABL+lam
        'profile_name': 'Uniform',
        'description': 'Uniform inlet, U_ref=1 m/s'
    },
    'U1_ABL_const': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_5_ABLc_U1',  
        'U_ref': 1.0,
        'profile_type': 1,
        'profile_name': 'ABL+constant',
        'description': 'ABL+constant no-slip top, U_ref=1 m/s'
    },
    'U1_ABL_lam': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_6_ABLcLam_U1', 
        'U_ref': 1.0,
        'profile_type': 3,
        'profile_name': 'ABL+laminar',
        'description': 'ABL+constant+laminar turbulence layer, U_ref=1 m/s'
    },
    'U1_ABL_free': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1', 
        'U_ref': 1.0,
        'profile_type': 2,
        'profile_name': 'ABL+free',
        'description': 'ABL+free top, U_ref=1 m/s'
    },
    'U5_uniform': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_8_Uniform_U5', 
        'U_ref': 5.1,
        'profile_type': 0,
        'profile_name': 'Uniform',
        'description': 'Uniform inlet, U_ref=5.1 m/s'
    },
    'U5_ABL_const': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_9_ABLc_U5',  
        'U_ref': 5.1,
        'profile_type': 1,
        'profile_name': 'ABL+constant',
        'description': 'ABL+constant no-slip top, U_ref=5.1 m/s'
    },
    'U5_ABL_lam': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_10_ABLcLam_U5_coded',  
        'U_ref': 5.1,
        'profile_type': 3,
        'profile_name': 'ABL+laminar',
        'description': 'ABL+constant+laminar turbulence layer, U_ref=5.1 m/s'
    },
    'U5_ABL_free': {
        'path': RAW_DATA_DIR / 'Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5',   
        'U_ref': 5.1,
        'profile_type': 2,
        'profile_name': 'ABL+free',
        'description': 'ABL+free top, U_ref=5.1 m/s'
    },
}

# X-locations for profile extraction
X_LOCATIONS = np.array([
    0.0000000,      # Inlet
    0.0375000,      # Non-disturbed flow
    0.0650000,      # In front of the cube
    0.0750000,      # Front face of the cube
    0.0825000,      # Top mid of the cube
    0.0903125,      # Just behind the cube
    0.1050000,      # Near wake:    1*Hc
    0.1200000,      # Further Wake: 2*Hc
    0.1350000,      # Further Wake: 3*Hc
    0.1500000,      # Far wake:     4*Hc
    0.3150000       # Outlet
])

# OpenFOAM extraction settings
TIME = 'latestTime'                  # Solution time: '10000', '2', 'latestTime' 
Y_LOCATION = None                    # Centerplane
TOLERANCE = 0.0012500                # 0.0012500 = coarse mesh dx
METHOD = 'closest'
N_BINS = 100

# Cube geometry (for reference)
H_CUBE = 0.015  # meters
X_CUBE_FRONT = 0.075
X_CUBE_BACK = 0.090


In [3]:
# ============================================
# MAIN LOADING FUNCTION
# ============================================

def load_all_cases():
    """
    Load all CFD cases and extract velocity profiles
    
    Returns:
        dict: all_cases_data with structure:
            {
                'case_id': {
                    'profiles': {x_loc: {'z': ..., 'U_x': ..., ...}},
                    'U_ref': float,
                    'profile_type': int,
                    'profile_name': str,
                    'description': str,
                    'path': str
                }
            }
    """
    
    all_cases_data = {}
    
    print("="*80)
    print("LOADING ALL OPENFOAM CASES")
    print("="*80)
    print(f"Total cases to load: {len(CASES)}")
    print(f"Profiles per case: {len(X_LOCATIONS)}")
    print(f"Expected total profiles: {len(CASES) * len(X_LOCATIONS)}")
    print("="*80 + "\n")
    
    for case_id, case_info in CASES.items():
        print("\n" + "="*80)
        print(f"CASE: {case_id}")
        print("="*80)
        print(f"Description: {case_info['description']}")
        print(f"Path: {case_info['path']}")
        print(f"U_ref: {case_info['U_ref']} m/s")
        print(f"Profile type: {case_info['profile_type']} ({case_info['profile_name']})")
        
        # Check if case directory exists
        if not os.path.exists(case_info['path']):
            print(f"⚠️  WARNING: Directory does not exist! Skipping...")
            continue
        
        print("-"*80)
        
        # Extract profiles at all x-locations
        profiles = {}
        success_count = 0
        fail_count = 0
        
        for x_loc in X_LOCATIONS:
            print(f"  x = {x_loc:.6f} m ... ", end='', flush=True)
            
            try:
                z_prof, Ux_prof, Uy_prof, Uz_prof = extract_profile_from_openfoam(
                    case_path=case_info['path'],
                    x_location=x_loc,
                    time=TIME,
                    y_location=Y_LOCATION,
                    tolerance=TOLERANCE,
                    method=METHOD,
                    n_bins=N_BINS
                )
                
                if z_prof is not None and len(z_prof) > 0:
                    profiles[x_loc] = {
                        'z': z_prof,
                        'U_x': Ux_prof,
                        'U_y': Uy_prof,
                        'U_z': Uz_prof,
                        'n_points': len(z_prof)
                    }
                    print(f"✓ ({len(z_prof)} points)")
                    success_count += 1
                else:
                    print(f"✗ FAILED (no data)")
                    fail_count += 1
                    
            except Exception as e:
                print(f"✗ ERROR: {str(e)}")
                fail_count += 1
        
        # Store case data
        all_cases_data[case_id] = {
            'profiles': profiles,
            'U_ref': case_info['U_ref'],
            'profile_type': case_info['profile_type'],
            'profile_name': case_info['profile_name'],
            'description': case_info['description'],
            'path': case_info['path'],
            'n_profiles_extracted': success_count,
        }
        
        print("-"*80)
        print(f"✓ Case loaded: {success_count}/{len(X_LOCATIONS)} profiles extracted")
        if fail_count > 0:
            print(f"⚠️  {fail_count} profiles failed")
    
    return all_cases_data



In [4]:
# ============================================
# VALIDATION & SUMMARY
# ============================================

def print_summary(all_cases_data):
    """Print summary statistics of loaded data"""
    
    print("\n" + "="*80)
    print("LOADING SUMMARY")
    print("="*80)
    
    total_cases = len(all_cases_data)
    total_profiles = sum(len(case['profiles']) for case in all_cases_data.values())
    total_points = sum(
        sum(prof['n_points'] for prof in case['profiles'].values())
        for case in all_cases_data.values()
    )
    
    print(f"Total cases loaded: {total_cases}")
    print(f"Total profiles extracted: {total_profiles}")
    print(f"Total data points: {total_points:,}")
    print(f"Average points per profile: {total_points / total_profiles:.0f}")
    
    print("\n" + "-"*80)
    print("Per-case summary:")
    print("-"*80)
    print(f"{'Case ID':<15} {'U_ref':<8} {'Profile':<15} {'N_profiles':<12} {'N_points':<10}")
    print("-"*80)
    
    for case_id, case_data in all_cases_data.items():
        n_profiles = len(case_data['profiles'])
        n_points = sum(prof['n_points'] for prof in case_data['profiles'].values())
        
        print(f"{case_id:<15} {case_data['U_ref']:<8.1f} {case_data['profile_name']:<15} "
              f"{n_profiles:<12} {n_points:<10,}")
    
    print("="*80)
    
    # Check for consistency
    print("\nConsistency checks:")
    
    # Check 1: All cases have same number of profiles?
    profile_counts = [len(case['profiles']) for case in all_cases_data.values()]
    if len(set(profile_counts)) == 1:
        print(f"  ✓ All cases have {profile_counts[0]} profiles (consistent)")
    else:
        print(f"  ⚠️  Cases have different numbers of profiles: {set(profile_counts)}")
    
    # Check 2: Same x-locations in all cases?
    all_x_locs = [set(case['profiles'].keys()) for case in all_cases_data.values()]
    if len(all_x_locs) > 0 and all(x_locs == all_x_locs[0] for x_locs in all_x_locs):
        print(f"  ✓ All cases have same x-locations")
    else:
        print(f"  ⚠️  Cases have different x-locations")
    
    # Check 3: Similar number of points per profile?
    all_point_counts = []
    for case in all_cases_data.values():
        for prof in case['profiles'].values():
            all_point_counts.append(prof['n_points'])
    
    if len(set(all_point_counts)) <= 3:  # Allow some variation
        print(f"  ✓ Point counts consistent (~{np.mean(all_point_counts):.0f} per profile)")
    else:
        print(f"  ⚠️  Point counts vary: {min(all_point_counts)} to {max(all_point_counts)}")
    
    print("="*80)

In [5]:
# ============================================
# MAIN EXECUTION
# ============================================

if __name__ == "__main__":
    
    print("\n" + "="*80)
    print(" "*25 + "STEP 1: DATA LOADING")
    print("="*80 + "\n")
    
    # Load all cases
    all_cases_data = load_all_cases()
    
    # Print summary
    print_summary(all_cases_data)
    
    # Save to pickle
#     output_file = 'all_cases_data.pkl'                  # Arbitrary name
    output_file = PROCESSED_DATA_DIR / 'all_cases_data.pkl'  
    print(f"\nSaving to: {output_file}")
    
    with open(output_file, 'wb') as f:
        pickle.dump(all_cases_data, f)
    
    print(f"✓ Saved successfully!")
    
    # Quick verification - reload and check
    print(f"\nVerifying saved file...")
    with open(output_file, 'rb') as f:
        loaded_data = pickle.load(f)
    
    print(f"  ✓ File readable")
    print(f"  ✓ Contains {len(loaded_data)} cases")
    print(f"  ✓ File size: {os.path.getsize(output_file) / 1024 / 1024:.2f} MB")
    
    print("\n" + "="*80)
    print("✓ STEP 1 COMPLETE!")
    print("="*80)
    print(f"\nOutput file: {output_file}")
    print(f"Ready for Step 2: Data Analysis")
    print("="*80 + "\n")


                         STEP 1: DATA LOADING

LOADING ALL OPENFOAM CASES
Total cases to load: 8
Profiles per case: 11
Expected total profiles: 88


CASE: U1_uniform
Description: Uniform inlet, U_ref=1 m/s
Path: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1
U_ref: 1.0 m/s
Profile type: 0 (Uniform)
--------------------------------------------------------------------------------
  x = 0.000000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1/10000/U
Reading 5,085,284 cells...

Extracted 20064 points near x=0.0
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] 

  x = 0.135000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1/10000/U
Reading 5,085,284 cells...

Extracted 40128 points near x=0.135
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [0.008497, 1.135690] m/s

✓ (55 points)
  x = 0.150000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_4_Uniform

Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_5_ABLc_U1/10000/U
Reading 5,085,284 cells...

Extracted 35925 points near x=0.0903125
  Reduced to 76 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 76
z range: [0.000313, 0.048125] m
U_x range: [-0.006532, 1.139310] m/s

✓ (76 points)
  x = 0.105000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_5_ABLc_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_5_ABLc_U1/10000/U
Reading 5,085,284 cells...

Extracted 40128 points near x=0.105
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [0.095983, 1.105690] m/s

✓ (55 points)
  x 

  x = 0.075000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_6_ABLcLam_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_6_ABLcLam_U1/10000/U
Reading 5,085,284 cells...

Extracted 42140 points near x=0.075
  Reduced to 76 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 76
z range: [0.000312, 0.048125] m
U_x range: [0.394556, 1.053190] m/s

✓ (76 points)
  x = 0.082500 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_6_ABLcLam_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_6_ABLcLam

Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1/10000/U
Reading 5,085,284 cells...

Extracted 20064 points near x=0.0
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [0.446617, 1.092380] m/s

✓ (55 points)
  x = 0.037500 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1/10000/U
Reading 5,085,284 cells...

Extracted 40128 points near x=0.0375
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [0.10

Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1/10000/U
Reading 5,085,284 cells...

Extracted 40128 points near x=0.15
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [0.060187, 1.129630] m/s

✓ (55 points)
  x = 0.315000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_7_ABLfreeTop_U1/10000/U
Reading 5,085,284 cells...

Extracted 20064 points near x=0.315
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [0.049460, 1.116560] m/s

✓ (55 

  x = 0.120000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_8_Uniform_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_8_Uniform_U5/10000/U
Reading 5,085,284 cells...

Extracted 40128 points near x=0.12
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [2.248740, 5.538100] m/s

✓ (55 points)
  x = 0.135000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_8_Uniform_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_8_Uniform_

Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_9_ABLc_U5/10000/U
Reading 5,085,284 cells...

Extracted 40988 points near x=0.0825
  Reduced to 77 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 77
z range: [0.000312, 0.048125] m
U_x range: [-0.978997, 5.801350] m/s

✓ (77 points)
  x = 0.090313 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_9_ABLc_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_9_ABLc_U5/10000/U
Reading 5,085,284 cells...

Extracted 35925 points near x=0.0903125
  Reduced to 76 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 76
z range: [0.000313, 0.048125] m
U_x range: [-0.075081, 5.691960] m/s

✓ (76 points)
  

  x = 0.065000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_10_ABLcLam_U5_coded/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_10_ABLcLam_U5_coded/10000/U
Reading 5,085,284 cells...

Extracted 41654 points near x=0.065
  Reduced to 65 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 65
z range: [0.000313, 0.048125] m
U_x range: [-3.508380, 5.191350] m/s

✓ (65 points)
  x = 0.075000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_10_ABLcLam_U5_coded/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmega

  x = 0.000000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5/10000/U
Reading 5,085,284 cells...

Extracted 20064 points near x=0.0
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [2.359760, 5.571130] m/s

✓ (55 points)
  x = 0.037500 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile

  x = 0.150000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5/10000/U
Reading 5,085,284 cells...

Extracted 40128 points near x=0.15
  Reduced to 55 points (closest at each height)

PROFILE EXTRACTED
Method: closest
Points in profile: 55
z range: [0.000313, 0.048125] m
U_x range: [1.100730, 5.706550] m/s

✓ (55 points)
  x = 0.315000 m ... ======================================================================
READING OPENFOAM DATA
Reading cell centers from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofile_11_ABLfreeTop_U5/10000/C
Reading 5,085,284 cells...
Reading field from: /home/vita/ML/DACI/Multi_Case_Data/data/raw/Cube_SIMPLE_kOmegaSST_Uprofil